# Bulk-update Feature Service metadata in ArcGIS Online from a CSV

Run this in an **ArcGIS Online Notebook** (or ArcGIS Pro). Steps:

1. Connect and search a group's items
2. Export the item list (with current metadata) to CSV
3. Edit the CSV: summary, tags, description, credits, terms of use
4. Upload the edited CSV and apply it

Blank cells are left unchanged, so you only need to fill in what you want to change.

### 1. Connect to your GIS

In [ ]:
from arcgis.gis import GIS
import pandas as pd

gis = GIS("home")

### 2. Search group items
Replace the group ID with your own (it's in the group page URL).

In [ ]:
GROUP_ID = "78aa453dfd244bdcb356128a62be9562"

search = gis.content.search(query=f"group:{GROUP_ID}", max_items=9999)
print(f"Found {len(search)} items")

### 3. Export items list
Includes the current metadata so you can edit it instead of starting from scratch.

In [ ]:
rows = []
for item in search:
    rows.append({
        "item_id": item.id,
        "item_name": item.title,
        "item_type": item.type,
        "summary": item.snippet or "",
        "tags": ", ".join(item.tags or []),
        "description": item.description or "",
        "credits": item.accessInformation or "",
        "terms_of_use": item.licenseInfo or "",
    })

df = pd.DataFrame(rows)
df.to_csv("/arcgis/home/Features.csv", index=False, encoding="utf-8-sig")
print("File Export Successful.")

### 4. Import metadata
Edit `Features.csv`, save it as `metadata.csv`, and upload it to `/arcgis/home/` (Files pane).

In [ ]:
# Load the CSV file. keep_default_na=False keeps empty cells as "" instead of NaN,
# which would otherwise be written to AGOL as the text "nan".
file_path = "/arcgis/home/metadata.csv"  # Update with the actual path
metadata_df = pd.read_csv(file_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")

In [ ]:
# CSV column -> ArcGIS item property.
# Note: "Credits (Attribution)" is stored as accessInformation, not "credits".
FIELDS = {
    "summary": "snippet",
    "tags": "tags",
    "description": "description",
    "credits": "accessInformation",
    "terms_of_use": "licenseInfo",
}

DRY_RUN = True  # set to False to actually update items


def update_item_metadata(row):
    item_id = row["item_id"].strip()
    try:
        item = gis.content.get(item_id)
        if not item:
            print(f"Item not found: {item_id}")
            return

        props = {}
        for column, prop in FIELDS.items():
            value = row.get(column, "").strip()
            if not value:
                continue  # blank cell = leave existing value alone
            props[prop] = [t.strip() for t in value.split(",") if t.strip()] if prop == "tags" else value

        if not props:
            print(f"Nothing to update: {item.title}")
        elif DRY_RUN:
            print(f"[dry-run] {item.title}: would set {', '.join(props)}")
        else:
            item.update(item_properties=props)
            print(f"Updated item: {item.title} ({item_id})")
    except Exception as e:
        print(f"Error updating item {item_id}: {e}")


# Iterate over the dataframe and update items
for _, row in metadata_df.iterrows():
    update_item_metadata(row)